# Explainable AI — DenseNet169 on ChestX-ray14 & CheXlocalize

**Model:** CheXpert-finetuned DenseNet169 (ChestX-ray14 → CheXpert pipeline)  
**Methods:** LayerCAM · FPN-LayerCAM · ScoreCAM · FPN-ScoreCAM · LIME (SLIC)  
**Input:** 100 ChestX-ray14 test samples + 100 CheXlocalize test samples  
**Output:** Per-image grid PNG — original + 5 multi-label composite heatmaps  

---
### FPN pyramid spec (DenseNet169)
| Block | Stride | Spatial | Weight |
|---|---|---|---|
| `features.denseblock1` | ×4 | 64×64 | 0.20 |
| `features.denseblock2` | ×8 | 32×32 | 0.35 |
| `features.denseblock4` | ×16 | 16×16 | 0.45 |

### Multi-label composite heatmap
- Every **detected** label (prob ≥ threshold) gets its own colour  
- Maps are alpha-blended over the original image  
- Legend shows label name · probability · ✓ (correct) / ✗ (missed/FP)

## 0. Imports & Reproducibility

In [ ]:
import warnings, random, gc
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import cv2
from PIL import Image
from skimage.segmentation import slic
from skimage.color import gray2rgb
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.models as models

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 1. Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# CheXpert-finetuned checkpoint (output of the CheXpert training notebook)
CHEXPERT_CKPT     = Path('./chexpert_output/best_model.pth')

# Manifests written by the two save_samples snippets
NIH_MANIFEST      = Path('./test_samples/manifest.csv')          # ChestX-ray14 100 images
CHEXLOCALIZE_MANIFEST = Path('./chexpert_output/test_samples/manifest.csv')  # CheXlocalize 100 images

# Output root — one sub-folder per dataset
XAI_OUTPUT        = Path('./xai_output')
NIH_XAI_DIR       = XAI_OUTPUT / 'chestxray14'
CHEX_XAI_DIR      = XAI_OUTPUT / 'chexlocalize'
for d in [NIH_XAI_DIR, CHEX_XAI_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Labels ────────────────────────────────────────────────────────────────────
# CheXpert label set (model output space — 14 classes)
LABELS: List[str] = [
    'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
    'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia',
    'Atelectasis', 'Pneumothorax', 'Effusion',
    'Pleural Other', 'Fracture', 'Support Devices', 'No Finding',
]
NUM_CLASSES = len(LABELS)

# ── Model & image settings ────────────────────────────────────────────────────
IMG_SIZE   = 224
THRESHOLD  = 0.5       # sigmoid threshold for positive prediction

# ── FPN pyramid configuration ─────────────────────────────────────────────────
FPN_LAYERS: List[Tuple[str, float]] = [
    ('features.denseblock1', 0.20),   # stride ×4,  spatial 64×64, low-level detail
    ('features.denseblock2', 0.35),   # stride ×8,  spatial 32×32, mid-level morphology
    ('features.denseblock4', 0.45),   # stride ×16, spatial 16×16, deep semantics
]

# ── LIME settings ─────────────────────────────────────────────────────────────
LIME_N_SEGMENTS  = 80     # SLIC superpixels
LIME_COMPACTNESS = 10     # SLIC compactness
LIME_N_SAMPLES   = 256    # perturbation samples — balance quality vs speed
LIME_TOP_FEATURES = 10    # number of superpixels to highlight

# ── Colour palette: 14 visually distinct colours for 14 classes ───────────────
# Designed for readability on grayscale X-ray backgrounds
LABEL_COLOURS = [
    '#FF3333', '#FF8C00', '#FFD700', '#7FFF00', '#00CED1',
    '#1E90FF', '#DA70D6', '#FF69B4', '#00FA9A', '#FF6347',
    '#40E0D0', '#BA55D3', '#F0E68C', '#87CEEB',
]
assert len(LABEL_COLOURS) == NUM_CLASSES

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print('Configuration ready.')
print(f'  Checkpoint     : {CHEXPERT_CKPT}')
print(f'  NIH manifest   : {NIH_MANIFEST}')
print(f'  CHEX manifest  : {CHEXLOCALIZE_MANIFEST}')
print(f'  Output root    : {XAI_OUTPUT}')

## 2. Model Loading

In [ ]:
def load_chexpert_model(
    ckpt_path:   Path,
    num_classes: int = NUM_CLASSES,
) -> nn.Module:
    """
    Rebuilds the CheXpert-finetuned DenseNet169 exactly as trained:
      backbone (DenseNet169) + Sequential(Dropout(0.5), Linear(1664, num_classes))
    Loads state_dict from checkpoint, sets eval mode.
    """
    model    = models.densenet169(weights=None)
    in_feat  = model.classifier.in_features   # 1664
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.5),
        nn.Linear(in_feat, num_classes),
    )
    ckpt      = torch.load(ckpt_path, map_location='cpu')
    state     = ckpt.get('state_dict', ckpt)
    model.load_state_dict(state, strict=True)
    model.eval()
    epoch_info = ckpt.get('epoch', '?')
    auc_info   = ckpt.get('val_auc', '?')
    print(f'Loaded CheXpert checkpoint — epoch {epoch_info}, val AUC {auc_info}')
    return model.to(DEVICE)


model = load_chexpert_model(CHEXPERT_CKPT)
# Keep a CPU copy for ScoreCAM (avoids repeated device transfers)
model_cpu = deepcopy(model).to('cpu').eval()

## 3. Image Utilities

In [ ]:
PREPROCESS = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def load_image_tensor(path: Path) -> Tuple[torch.Tensor, np.ndarray]:
    """
    Returns:
      tensor  : (1, 3, H, W) float32 normalised, on DEVICE
      img_rgb : (H, W, 3)    uint8 numpy array for visualisation
    """
    pil     = Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    img_rgb = np.array(pil, dtype=np.uint8)
    tensor  = PREPROCESS(pil).unsqueeze(0).to(DEVICE)
    return tensor, img_rgb


def denormalise(tensor: torch.Tensor) -> np.ndarray:
    """Reverse ImageNet normalisation → uint8 HWC."""
    mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    std  = torch.tensor(IMAGENET_STD ).view(3,1,1)
    img  = (tensor.squeeze(0).cpu() * std + mean).clamp(0,1)
    return (img.permute(1,2,0).numpy() * 255).astype(np.uint8)


def normalise_cam(cam: np.ndarray) -> np.ndarray:
    """Min-max normalise a CAM map to [0, 1]."""
    mn, mx = cam.min(), cam.max()
    if mx - mn < 1e-8:
        return np.zeros_like(cam, dtype=np.float32)
    return ((cam - mn) / (mx - mn)).astype(np.float32)


def hex_to_rgb(h: str) -> Tuple[int, int, int]:
    h = h.lstrip('#')
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))


@torch.no_grad()
def get_predictions(
    model:  nn.Module,
    tensor: torch.Tensor,
) -> np.ndarray:
    """Returns sigmoid probabilities (NUM_CLASSES,)."""
    return model(tensor).sigmoid().squeeze(0).cpu().numpy()

## 4. Hook Infrastructure (shared by LayerCAM & ScoreCAM)

In [ ]:
class ActivationGradientHook:
    """
    Registers forward + backward hooks on a target layer.
    Stores the last activation map and its gradient.
    """

    def __init__(self, layer: nn.Module) -> None:
        self.activation: Optional[torch.Tensor] = None
        self.gradient:   Optional[torch.Tensor] = None
        self._fwd = layer.register_forward_hook(self._save_activation)
        self._bwd = layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out) -> None:
        self.activation = out.detach()

    def _save_gradient(self, module, grad_in, grad_out) -> None:
        self.gradient = grad_out[0].detach()

    def remove(self) -> None:
        self._fwd.remove()
        self._bwd.remove()


def get_layer_by_name(model: nn.Module, name: str) -> nn.Module:
    """Traverse dot-separated module path, e.g. 'features.denseblock1'."""
    parts = name.split('.')
    m = model
    for p in parts:
        m = getattr(m, p)
    return m

## 5. LayerCAM

LayerCAM (Jiang et al. 2021): element-wise product of activation and its ReLU'd gradient,  
then spatially averaged across channels. Applied to `features.denseblock4`.

In [ ]:
# Target layer for single-layer CAM methods
LAYERCAM_TARGET = 'features.denseblock4'


def layercam_single_class(
    model:      nn.Module,
    tensor:     torch.Tensor,
    class_idx:  int,
    layer_name: str = LAYERCAM_TARGET,
) -> np.ndarray:
    """
    Computes LayerCAM for a single class.
    Returns upsampled map (IMG_SIZE, IMG_SIZE) in [0, 1].
    """
    layer = get_layer_by_name(model, layer_name)
    hook  = ActivationGradientHook(layer)

    model.zero_grad()
    logits = model(tensor)                        # (1, NUM_CLASSES)
    logits[0, class_idx].backward()

    act  = hook.activation.squeeze(0)             # (C, h, w)
    grad = hook.gradient.squeeze(0)               # (C, h, w)
    hook.remove()

    # LayerCAM: element-wise product with ReLU'd gradient
    cam  = (act * F.relu(grad)).sum(dim=0)        # (h, w)
    cam  = F.relu(cam).cpu().numpy()

    # Upsample to input resolution
    cam  = cv2.resize(cam, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    return normalise_cam(cam)


def layercam_multilabel(
    model:     nn.Module,
    tensor:    torch.Tensor,
    class_ids: List[int],
    layer_name: str = LAYERCAM_TARGET,
) -> Dict[int, np.ndarray]:
    """Returns {class_idx: cam_map} for each class in class_ids."""
    return {
        cid: layercam_single_class(model, tensor, cid, layer_name)
        for cid in class_ids
    }

## 6. FPN-LayerCAM

Computes LayerCAM independently at three pyramid levels then fuses via weighted sum after upsampling to 224×224.

In [ ]:
def fpn_layercam_single_class(
    model:      nn.Module,
    tensor:     torch.Tensor,
    class_idx:  int,
    fpn_layers: List[Tuple[str, float]] = FPN_LAYERS,
) -> np.ndarray:
    """
    FPN-LayerCAM for one class:
      1. For each (layer_name, weight) in fpn_layers:
         a. Forward + backward pass to get activations and gradients.
         b. Compute LayerCAM at that layer's native resolution.
         c. Upsample to IMG_SIZE × IMG_SIZE.
         d. Scale by pyramid weight.
      2. Sum scaled maps and normalise to [0, 1].

    NOTE: Each pyramid level requires its own backward pass because
    DenseNet169's skip connections mean gradients from a single pass
    accumulate across blocks and the individual layer signals interfere.
    Separate passes give clean, decoupled per-level gradient estimates.
    """
    fused = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32)

    for layer_name, weight in fpn_layers:
        layer = get_layer_by_name(model, layer_name)
        hook  = ActivationGradientHook(layer)

        model.zero_grad()
        logits = model(tensor)
        logits[0, class_idx].backward(retain_graph=True)

        act  = hook.activation.squeeze(0)          # (C, h, w)
        grad = hook.gradient.squeeze(0)            # (C, h, w)
        hook.remove()

        cam  = (act * F.relu(grad)).sum(dim=0)
        cam  = F.relu(cam).cpu().numpy()
        cam  = cv2.resize(cam, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        cam  = normalise_cam(cam)
        fused += weight * cam

    model.zero_grad()
    return normalise_cam(fused)


def fpn_layercam_multilabel(
    model:     nn.Module,
    tensor:    torch.Tensor,
    class_ids: List[int],
) -> Dict[int, np.ndarray]:
    return {
        cid: fpn_layercam_single_class(model, tensor, cid)
        for cid in class_ids
    }

## 7. ScoreCAM

ScoreCAM (Wang et al. 2020): gradient-free. Each activation channel is used as a mask  
applied to the input; the forward-pass score change gives the channel weight.  
Computationally expensive — we batch the masked inputs to manage VRAM.

In [ ]:
# ScoreCAM target layer (same as LayerCAM for fair comparison)
SCORECAM_TARGET = 'features.denseblock4'
SCORECAM_BATCH  = 16   # number of masked inputs processed at once; reduce if OOM


def _get_activations(
    model:      nn.Module,
    tensor:     torch.Tensor,
    layer_name: str,
) -> torch.Tensor:
    """Returns activation map (C, h, w) for the given layer."""
    acts = {}
    layer = get_layer_by_name(model, layer_name)
    hook  = layer.register_forward_hook(lambda m, i, o: acts.update({'v': o.detach()}))
    with torch.no_grad():
        model(tensor)
    hook.remove()
    return acts['v'].squeeze(0)   # (C, h, w)


def scorecam_single_class(
    model:      nn.Module,
    tensor:     torch.Tensor,
    class_idx:  int,
    layer_name: str   = SCORECAM_TARGET,
    batch_size: int   = SCORECAM_BATCH,
) -> np.ndarray:
    """
    ScoreCAM for one class. Runs entirely in no_grad — memory-safe.
    Each activation channel is upsampled, normalised to [0,1], then used
    as a multiplicative mask on the input image.
    The model score for class_idx on the masked image = channel weight.
    """
    activations = _get_activations(model, tensor, layer_name)  # (C, h, w)
    n_channels  = activations.shape[0]

    # Baseline score: forward pass on the input zeroed out (black image)
    with torch.no_grad():
        baseline_score = model(
            torch.zeros_like(tensor)
        ).sigmoid()[0, class_idx].item()

    # Build all masked inputs — process in batches to bound VRAM
    channel_weights = np.zeros(n_channels, dtype=np.float32)

    for batch_start in range(0, n_channels, batch_size):
        batch_end  = min(batch_start + batch_size, n_channels)
        masks_list = []

        for c in range(batch_start, batch_end):
            ch_map = activations[c].cpu().numpy()            # (h, w)
            ch_up  = cv2.resize(
                ch_map, (IMG_SIZE, IMG_SIZE),
                interpolation=cv2.INTER_LINEAR
            )
            # Normalise channel map to [0, 1] so it acts as a soft mask
            mn, mx = ch_up.min(), ch_up.max()
            if mx - mn < 1e-8:
                ch_up = np.zeros_like(ch_up)
            else:
                ch_up = (ch_up - mn) / (mx - mn)
            mask = torch.from_numpy(ch_up).float().to(DEVICE)  # (H, W)
            masked_input = tensor * mask.unsqueeze(0).unsqueeze(0)  # broadcast
            masks_list.append(masked_input)

        batch_tensor = torch.cat(masks_list, dim=0)           # (B, 3, H, W)
        with torch.no_grad():
            scores = model(batch_tensor).sigmoid()[:, class_idx].cpu().numpy()

        channel_weights[batch_start:batch_end] = scores - baseline_score

    # Weighted sum of activation maps
    weights_t = torch.from_numpy(channel_weights).to(activations.device)  # (C,)
    cam  = (F.relu(weights_t).unsqueeze(-1).unsqueeze(-1) * activations).sum(0)
    cam  = F.relu(cam).cpu().numpy()
    cam  = cv2.resize(cam, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    return normalise_cam(cam)


def scorecam_multilabel(
    model:     nn.Module,
    tensor:    torch.Tensor,
    class_ids: List[int],
) -> Dict[int, np.ndarray]:
    return {
        cid: scorecam_single_class(model, tensor, cid)
        for cid in class_ids
    }

## 8. FPN-ScoreCAM

Applies ScoreCAM independently at three pyramid levels and fuses with the same weights as FPN-LayerCAM.

In [ ]:
def fpn_scorecam_single_class(
    model:      nn.Module,
    tensor:     torch.Tensor,
    class_idx:  int,
    fpn_layers: List[Tuple[str, float]] = FPN_LAYERS,
    batch_size: int = SCORECAM_BATCH,
) -> np.ndarray:
    """
    FPN-ScoreCAM: compute ScoreCAM at each pyramid level, upsample,
    apply FPN weights, sum, and normalise.
    """
    fused = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32)
    for layer_name, weight in fpn_layers:
        cam   = scorecam_single_class(model, tensor, class_idx,
                                      layer_name=layer_name,
                                      batch_size=batch_size)
        fused += weight * cam
    return normalise_cam(fused)


def fpn_scorecam_multilabel(
    model:     nn.Module,
    tensor:    torch.Tensor,
    class_ids: List[int],
) -> Dict[int, np.ndarray]:
    return {
        cid: fpn_scorecam_single_class(model, tensor, cid)
        for cid in class_ids
    }

## 9. LIME (SLIC superpixels)

LIME (Ribeiro et al. 2016): perturbs superpixel segments, trains a local linear surrogate,  
and identifies which segments most influence each class prediction.

In [ ]:
def lime_multilabel(
    model:          nn.Module,
    img_rgb:        np.ndarray,
    class_ids:      List[int],
    n_segments:     int = LIME_N_SEGMENTS,
    compactness:    float = LIME_COMPACTNESS,
    n_samples:      int = LIME_N_SAMPLES,
    top_features:   int = LIME_TOP_FEATURES,
) -> Dict[int, np.ndarray]:
    """
    Model-agnostic LIME implementation without the lime library dependency.

    Algorithm:
      1. Segment image with SLIC → S superpixels.
      2. Sample n_samples binary vectors z ∈ {0,1}^S (active/inactive segments).
      3. For each z: construct perturbed image (inactive → mean colour).
      4. Get model sigmoid scores for each perturbed image.
      5. For each class: fit weighted Ridge regression (y = Xw) where
         sample weight = cosine similarity to the all-ones vector (proximity).
      6. The top `top_features` positive coefficients indicate influential segments.
      7. Build a spatial heatmap: coefficient value per pixel.

    Returns {class_idx: heatmap (IMG_SIZE, IMG_SIZE) in [0, 1]}.
    Uses the CPU model copy to avoid accumulating GPU memory across
    the large perturbation batch.
    """
    from sklearn.linear_model import Ridge
    from sklearn.preprocessing import normalize as sk_normalise

    # ── 1. Segmentation ────────────────────────────────────────────────────────
    segments = slic(
        img_rgb, n_segments=n_segments,
        compactness=compactness, start_label=0
    )
    n_segs      = segments.max() + 1
    mean_colour = img_rgb.mean(axis=(0,1))   # fallback fill for inactive segments

    # ── 2. Sample binary masks ─────────────────────────────────────────────────
    rng       = np.random.default_rng(SEED)
    Z         = rng.integers(0, 2, size=(n_samples, n_segs)).astype(np.float32)
    Z[0, :]   = 1.0   # first sample = original image (all segments active)

    # ── 3. Build perturbed images ──────────────────────────────────────────────
    perturbed_tensors = []
    for z in Z:
        pimg = img_rgb.copy().astype(np.float32)
        for seg_id in range(n_segs):
            if z[seg_id] == 0:
                pimg[segments == seg_id] = mean_colour
        pil_p = Image.fromarray(pimg.astype(np.uint8))
        perturbed_tensors.append(PREPROCESS(pil_p))

    # ── 4. Score perturbed images (CPU, batched) ───────────────────────────────
    batch = torch.stack(perturbed_tensors)              # (n_samples, 3, H, W)
    PRED_BATCH = 32
    all_scores = []
    with torch.no_grad():
        for i in range(0, len(batch), PRED_BATCH):
            scores = model_cpu(batch[i:i+PRED_BATCH]).sigmoid()
            all_scores.append(scores)
    scores = torch.cat(all_scores).numpy()              # (n_samples, NUM_CLASSES)

    # ── 5. Proximity weights (cosine similarity to z=all-ones) ────────────────
    ones      = np.ones((1, n_segs), dtype=np.float32)
    Z_norm    = sk_normalise(Z, norm='l2')
    ones_norm = sk_normalise(ones, norm='l2')
    weights   = (Z_norm @ ones_norm.T).squeeze(1)       # (n_samples,)
    weights   = np.clip(weights, 0, None)

    # ── 6. Fit Ridge per class, build spatial heatmaps ────────────────────────
    heatmaps: Dict[int, np.ndarray] = {}
    for cid in class_ids:
        y      = scores[:, cid]                         # (n_samples,)
        ridge  = Ridge(alpha=1.0, fit_intercept=True)
        ridge.fit(Z, y, sample_weight=weights)
        coefs  = ridge.coef_                            # (n_segs,)

        # Keep only top positive coefficients
        coefs_clipped = np.maximum(coefs, 0)
        if coefs_clipped.sum() > 0:
            top_ids = np.argsort(coefs_clipped)[-top_features:]
            mask_coefs = np.zeros_like(coefs_clipped)
            mask_coefs[top_ids] = coefs_clipped[top_ids]
        else:
            mask_coefs = coefs_clipped

        # Map back to pixel space
        hmap = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32)
        for seg_id in range(n_segs):
            hmap[segments == seg_id] = mask_coefs[seg_id]

        heatmaps[cid] = normalise_cam(hmap)

    return heatmaps

## 10. Multi-Label Composite Heatmap Renderer

In [ ]:
def blend_multilabel_heatmap(
    img_rgb:   np.ndarray,
    cam_maps:  Dict[int, np.ndarray],
    probs:     np.ndarray,
    gt_labels: np.ndarray,
    alpha:     float = 0.55,
) -> np.ndarray:
    """
    Blends per-class CAM maps onto img_rgb.

    Each class gets its own colour from LABEL_COLOURS.
    Per-pixel: colour is the class with the highest weighted cam value.
    The alpha mask is the max cam value across all active classes.

    Returns composited RGB image (H, W, 3) uint8.
    """
    H, W = IMG_SIZE, IMG_SIZE
    canvas       = img_rgb.astype(np.float32)
    colour_layer = np.zeros((H, W, 3), dtype=np.float32)
    weight_layer = np.zeros((H, W),    dtype=np.float32)

    for cid, cam in cam_maps.items():
        rgb   = np.array(hex_to_rgb(LABEL_COLOURS[cid]), dtype=np.float32)
        # Pixels where this class has higher cam than current winner
        dominant = cam > weight_layer
        colour_layer[dominant] = rgb
        weight_layer[dominant] = cam[dominant]

    # Alpha blend: higher cam → more colour visible
    alpha_map = (weight_layer * alpha)[..., np.newaxis]   # (H, W, 1)
    composite = canvas * (1 - alpha_map) + colour_layer * alpha_map
    return np.clip(composite, 0, 255).astype(np.uint8)


def make_legend_patches(
    class_ids:  List[int],
    probs:      np.ndarray,
    gt_labels:  np.ndarray,
) -> List[mpatches.Patch]:
    """
    Builds matplotlib legend patches.
    Tick (✓) if ground-truth positive, cross (✗) if false positive.
    """
    patches = []
    for cid in class_ids:
        colour  = LABEL_COLOURS[cid]
        symbol  = '\u2713' if gt_labels[cid] == 1 else '\u2717'
        label   = f'{symbol} {LABELS[cid]}  {probs[cid]:.2f}'
        patches.append(mpatches.Patch(color=colour, label=label))
    return patches

## 11. Full XAI Pipeline for a Single Image

In [ ]:
def run_xai_single(
    model:      nn.Module,
    img_path:   Path,
    gt_labels:  np.ndarray,
    output_path: Path,
    threshold:  float = THRESHOLD,
) -> None:
    """
    Full pipeline for one image:
      1. Load image & get predictions.
      2. Determine active (detected) classes.
      3. Compute all 5 XAI methods.
      4. Render multi-label composite heatmaps.
      5. Save a 2×3 grid figure:
           [Original] [LayerCAM] [FPN-LayerCAM]
           [ScoreCAM] [FPN-ScoreCAM] [LIME]
    """
    tensor, img_rgb = load_image_tensor(img_path)
    probs           = get_predictions(model, tensor)

    # Classes where model fires above threshold
    pred_ids = [i for i, p in enumerate(probs) if p >= threshold]

    # Fall back to top-1 if model fires on nothing
    if not pred_ids:
        pred_ids = [int(np.argmax(probs))]

    # ── Compute XAI maps ──────────────────────────────────────────────────────
    lc_maps   = layercam_multilabel(model, tensor, pred_ids)
    fpn_lc    = fpn_layercam_multilabel(model, tensor, pred_ids)
    sc_maps   = scorecam_multilabel(model, tensor, pred_ids)
    fpn_sc    = fpn_scorecam_multilabel(model, tensor, pred_ids)
    lime_maps = lime_multilabel(model, img_rgb, pred_ids)

    # ── Composite overlays ────────────────────────────────────────────────────
    overlay_lc    = blend_multilabel_heatmap(img_rgb, lc_maps,  probs, gt_labels)
    overlay_fpnlc = blend_multilabel_heatmap(img_rgb, fpn_lc,   probs, gt_labels)
    overlay_sc    = blend_multilabel_heatmap(img_rgb, sc_maps,  probs, gt_labels)
    overlay_fpnsc = blend_multilabel_heatmap(img_rgb, fpn_sc,   probs, gt_labels)
    overlay_lime  = blend_multilabel_heatmap(img_rgb, lime_maps, probs, gt_labels)

    legend_patches = make_legend_patches(pred_ids, probs, gt_labels)

    # ── Build 2×3 figure grid ─────────────────────────────────────────────────
    panels = [
        (img_rgb,        'Original'),
        (overlay_lc,     'LayerCAM'),
        (overlay_fpnlc,  'FPN-LayerCAM'),
        (overlay_sc,     'ScoreCAM'),
        (overlay_fpnsc,  'FPN-ScoreCAM'),
        (overlay_lime,   'LIME (SLIC)'),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(13, 9))
    axes = axes.flatten()

    for ax, (panel_img, title) in zip(axes, panels):
        ax.imshow(panel_img)
        ax.set_title(title, fontsize=11, pad=6)
        ax.axis('off')

    # Legend placed below all panels
    fig.legend(
        handles   = legend_patches,
        loc       = 'lower center',
        ncol      = min(len(pred_ids), 4),
        fontsize  = 9,
        framealpha= 0.9,
        title     = 'Detected labels  (✓ correct  ✗ false positive)',
        title_fontsize = 9,
        bbox_to_anchor = (0.5, -0.01),
    )

    fig.suptitle(
        f'{img_path.name}',
        fontsize=10, y=1.01
    )
    plt.tight_layout(rect=[0, 0.08, 1, 1])
    plt.savefig(output_path, dpi=130, bbox_inches='tight')
    plt.close(fig)

    # Free GPU memory between images
    torch.cuda.empty_cache()
    gc.collect()

## 12. Batch Runner

In [ ]:
def run_xai_batch(
    model:        nn.Module,
    manifest_path: Path,
    image_dir:    Path,
    output_dir:   Path,
    dataset_name: str,
    label_cols:   List[str] = LABELS,
) -> None:
    """
    Runs the full XAI pipeline on every image listed in manifest_path.

    manifest_path : CSV with 'filename' column and one column per label.
    image_dir     : directory where the saved sample images live.
    output_dir    : where to write the output grid PNGs.
    dataset_name  : used for progress bar label only.

    Missing label columns default to 0 (handles the cross-dataset case
    where NIH manifest uses NIH labels but model outputs CheXpert labels).
    """
    manifest = pd.read_csv(manifest_path)
    output_dir.mkdir(parents=True, exist_ok=True)

    skipped = []
    for _, row in tqdm(manifest.iterrows(), total=len(manifest),
                       desc=f'XAI [{dataset_name}]'):
        fname    = row['filename']
        img_path = image_dir / fname

        if not img_path.exists():
            skipped.append(fname)
            continue

        # Build ground-truth vector aligned to model's LABELS
        gt = np.array([
            float(row[lbl]) if lbl in row.index else 0.0
            for lbl in label_cols
        ], dtype=np.float32)

        out_path = output_dir / f'{Path(fname).stem}_xai.png'
        if out_path.exists():
            continue   # resume-friendly: skip already-processed images

        try:
            run_xai_single(model, img_path, gt, out_path)
        except Exception as e:
            print(f'  ERROR on {fname}: {e}')
            skipped.append(fname)

    print(f'\n{dataset_name} complete.')
    print(f'  Saved  : {len(manifest) - len(skipped)} / {len(manifest)} images')
    if skipped:
        print(f'  Skipped: {skipped}')

## 13. Run — ChestX-ray14 Test Samples

In [ ]:
# The CheXpert-finetuned model outputs CheXpert labels.
# The NIH manifest contains NIH label columns (different vocabulary).
# run_xai_batch handles missing columns gracefully (→ 0),
# so the ✓/✗ marks in the legend reflect NIH ground-truth where available
# and are absent (treated as unknown) for CheXpert-only labels.

run_xai_batch(
    model         = model,
    manifest_path = NIH_MANIFEST,
    image_dir     = NIH_MANIFEST.parent,
    output_dir    = NIH_XAI_DIR,
    dataset_name  = 'ChestX-ray14',
)

## 14. Run — CheXlocalize Test Samples

In [ ]:
run_xai_batch(
    model         = model,
    manifest_path = CHEXLOCALIZE_MANIFEST,
    image_dir     = CHEXLOCALIZE_MANIFEST.parent,
    output_dir    = CHEX_XAI_DIR,
    dataset_name  = 'CheXlocalize',
)

## 15. Spot-Check — Display One Result Inline

In [ ]:
def display_sample(output_dir: Path, idx: int = 0) -> None:
    """Display the idx-th saved grid from output_dir inline in the notebook."""
    pngs = sorted(output_dir.glob('*_xai.png'))
    if not pngs:
        print('No output images found yet.')
        return
    chosen = pngs[idx % len(pngs)]
    img    = Image.open(chosen)
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.imshow(np.array(img))
    ax.axis('off')
    ax.set_title(chosen.name, fontsize=10)
    plt.tight_layout()
    plt.show()

print('ChestX-ray14 sample:')
display_sample(NIH_XAI_DIR, idx=0)

print('CheXlocalize sample:')
display_sample(CHEX_XAI_DIR, idx=0)

## 16. Output Summary

In [ ]:
nih_pngs  = list(NIH_XAI_DIR.glob('*_xai.png'))
chex_pngs = list(CHEX_XAI_DIR.glob('*_xai.png'))

print('=' * 56)
print('  XAI Output Summary')
print('=' * 56)
print(f'  ChestX-ray14  grids : {len(nih_pngs):>4}  →  {NIH_XAI_DIR.resolve()}')
print(f'  CheXlocalize  grids : {len(chex_pngs):>4}  →  {CHEX_XAI_DIR.resolve()}')
print(f'  Total               : {len(nih_pngs)+len(chex_pngs):>4}')
print('=' * 56)
print('\nGrid layout per image:')
print('  Row 1: Original | LayerCAM       | FPN-LayerCAM')
print('  Row 2: ScoreCAM | FPN-ScoreCAM   | LIME (SLIC)')
print('\nLegend key:')
print('  ✓  ground-truth positive label (correctly detected)')
print('  ✗  false positive (model detected, not in ground-truth)')